<a href="https://colab.research.google.com/github/GUNAPILLCO/neural_profit/blob/main/stage_07_model_analysis/stage_07a_model_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Stage_07_00 – Model Analysis**

In [1]:
import pandas as pd
from pathlib import Path
import os

In [2]:
from google.colab import drive
drive.mount('/content/drive')

# --------------------------------------------------
# Ruta base en Google Drive
# --------------------------------------------------
# Ajuste si su estructura cambia
DRIVE_DIR =   Path(os.environ.get("DRIVE_DIR", "/content/drive/MyDrive/neural_profit/"))

Mounted at /content/drive


# **1. Métricas de SEQ2ONE**

In [3]:
import pandas as pd
from pathlib import Path

def load_all_seq2one_metrics(
    *,
    models: list[str],
    base_dir: str = "/content/drive/MyDrive/neural_profit/metrics/seq2one_metrics",
) -> pd.DataFrame:
    """
    Carga métricas seq2one para múltiples modelos y
    mantiene solo columnas estándar comparables.
    """

    cols = [
        "model", "split", "window_size", "target",
        "horizon_min", "MAE", "RMSE", "R2", "DA"
    ]

    dfs = []

    base_path = Path(base_dir)

    for name in models:
        path = base_path / f"seq2one_{name}_metrics.parquet"

        if not path.exists():
            continue

        df = pd.read_parquet(path)

        # Mantener solo columnas deseadas si existen
        keep_cols = [c for c in cols if c in df.columns]
        df = df[keep_cols].copy()

        dfs.append(df)

    if not dfs:
        return pd.DataFrame(columns=cols)

    df_all = pd.concat(dfs, ignore_index=True)

    # Orden consistente
    df_all = (
        df_all
        .sort_values(["model", "window_size", "target", "split"])
        .reset_index(drop=True)
    )

    return df_all


In [4]:
#models_seq2one = ['naive', 'ridge', 'lasso', 'mlp', 'gru', 'lstm', 'tcn', 'transformer']
models_seq2one = ['naive', 'ridge', 'lasso', 'mlp', 'gru', 'lstm', 'transformer']

df_seq2one_all = load_all_seq2one_metrics(
    models=models_seq2one,
    base_dir="/content/drive/MyDrive/neural_profit/metrics/seq2one_metrics"
)

In [5]:
df_seq2one_all

,model,split,window_size,target,horizon_min,MAE,RMSE,R2,DA
0,gru,test,30,delta_60,60,53.763203,83.362923,-0.001191,0.472136
1,gru,valid,30,delta_60,60,35.524135,50.330595,-0.004840,0.475907
2,gru,test,30,delta_90,90,67.621347,103.995571,-0.000133,0.475719
3,gru,valid,30,delta_90,90,44.602196,63.079182,-0.007723,0.480341
4,gru,test,30,ret_60,60,0.003187,0.006342,-1.193853,0.496818
...,...,...,...,...,...,...,...,...,...
315,transformer,valid,180,delta_90,90,49.143029,67.458543,-0.007562,0.466113
316,transformer,test,180,ret_60,60,0.007926,0.012558,-5.833867,0.499461
317,transformer,valid,180,ret_60,60,0.008658,0.010358,-10.963723,0.534856
318,transformer,test,180,ret_90,90,0.005890,0.009907,-1.774780,0.494162


## **1.1. Selección del tipo de target**

### **1.1.1. Criterios técnicos**

**Criterios primarios**

- R² > 0 de forma consistente entre modelos.
- MAE estable y razonable.
- Directional Accuracy (DA) claramente superior a 0.55.
- Baja varianza de desempeño entre modelos.

**Observación preliminar**

- `delta_60` y `delta_90` presentan R² positivos y razonables.
- `ret_60` y `ret_90` muestran R² negativos en la mayoría de los casos.

**Decisión metodológica**

Primero se debe seleccionar entre `delta` y `ret`.

Si el objetivo es obtener una señal predictiva explotable, el target elegido debe:

- Presentar R² positivo en la mayoría de los modelos.
- Tener mayor DA promedio.
- Mostrar menor dispersión de resultados entre arquitecturas.

Preliminarmente, el candidato más sólido parece ser `delta_60`.

**Criterio de decisión: uso de R² como métrica principal**

**1. Qué mide R²**

R² mide la **proporción de varianza explicada** por el modelo respecto a un baseline constante.

En regresión:

R² = 1 − (MSE_model / MSE_baseline)

Donde el baseline es predecir siempre la media del target.

Interpretación:

- R² > 0 → el modelo mejora al baseline.
- R² = 0 → el modelo es equivalente al baseline.
- R² < 0 → el modelo es peor que el baseline.

Por lo tanto, R² es una métrica estructural que indica si existe capacidad explicativa real.

---

**2. Relevancia para este problema**

En esta etapa el objetivo no es aún maximizar PnL, sino:

> Evaluar si el target es predecible.

R² responde directamente a esa pregunta.

MAE y RMSE solo miden magnitud del error absoluto, pero no indican si el modelo mejora significativamente respecto a un baseline simple.

Un modelo puede tener MAE bajo y aun así no explicar varianza relevante si el target tiene poca dispersión.

---

**3. Interpretación de R² en series financieras**

En problemas financieros:

- R² ≈ 5% ya es interesante.
- R² ≈ 20–30% es muy fuerte.
- R² negativo implica ausencia de señal explotable.

Usar R² como criterio principal equivale a preguntar:

> ¿Existe señal estructural o estamos modelando ruido?

---

**4. Por qué MAE no es la métrica principal**

Limitaciones del MAE:

- Depende de la escala del target.
- No es relativo a un baseline.
- No permite comparar fácilmente targets con distinta varianza.

Ejemplo:

- delta_60 puede tener MAE = 25
- ret_60 puede tener MAE = 0.002

No son directamente comparables.

R² sí permite comparación transversal entre targets.

---

**5. Por qué DA no puede ser el criterio central**

Directional Accuracy (DA):

- Ignora magnitud del error.
- Puede inflarse si el target tiene sesgo estructural.
- No penaliza errores grandes.

DA es útil como métrica complementaria, pero no como criterio principal de selección.

---

**6. Orden metodológico en esta etapa**

Para la selección del tipo de target:

1. R² → evaluar capacidad explicativa.
2. DA → validar señal direccional.
3. MAE → analizar estabilidad del error.
4. Varianza del desempeño → evaluar robustez entre modelos.

---

**Conclusión**

R² se utiliza como métrica principal porque:

- Es relativa al baseline.
- Mide capacidad explicativa real.
- Permite comparar targets con distinta escala.
- Responde a la pregunta fundamental: si existe señal estructural en el target.

### **1.1.2. Implementación**

In [6]:
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

In [7]:
import pandas as pd
import numpy as np

# -------------------------------------------------
# Configuración
# -------------------------------------------------
DF = df_seq2one_all.copy()
SPLIT = "valid"

# Filtrar solo VALID
df_valid = DF[DF["split"] == SPLIT].copy()

In [8]:
import pandas as pd
import numpy as np

# Requiere df_valid con columnas:
# ['model','split','window_size','target','horizon_min','MAE','RMSE','R2','DA']

required = {"model","split","window_size","target","horizon_min","MAE","RMSE","R2","DA"}
missing = required - set(df_valid.columns)
if missing:
    raise ValueError(f"df_valid no tiene columnas requeridas: {sorted(missing)}")

df = df_valid.copy()
df = df[df["split"].astype(str).str.lower().eq("valid")].copy()

# Normalización mínima
df["target"] = df["target"].astype(str).str.lower().str.strip()

# Tipo de target: delta vs ret
df["target_type"] = np.where(df["target"].str.startswith("delta"), "delta",
                      np.where(df["target"].str.startswith("ret"), "ret", "other"))

# Flags de criterios primarios
df["r2_pos"] = df["R2"] > 0
df["da_gt_055"] = df["DA"] > 0.55

def summarize(g: pd.DataFrame) -> pd.Series:
    return pd.Series({
        "n_rows": len(g),
        "n_models": g["model"].nunique(),
        "n_windows": g["window_size"].nunique(),

        "R2_mean": g["R2"].mean(),
        "R2_median": g["R2"].median(),
        "R2_std": g["R2"].std(ddof=0),
        "R2_share_pos": g["r2_pos"].mean(),

        "DA_mean": g["DA"].mean(),
        "DA_median": g["DA"].median(),
        "DA_std": g["DA"].std(ddof=0),
        "DA_share_gt_055": g["da_gt_055"].mean(),

        "MAE_mean": g["MAE"].mean(),
        "MAE_median": g["MAE"].median(),
        "MAE_std": g["MAE"].std(ddof=0),
    })

def fmt_table(t: pd.DataFrame) -> pd.DataFrame:
    out = t.copy()
    # porcentajes
    for c in ["R2_share_pos", "DA_share_gt_055"]:
        if c in out.columns:
            out[c] = (out[c] * 100).round(1).astype(str) + "%"
    # redondeo numérico
    for c in out.columns:
        if c not in ["target","target_type","model","n_rows","n_models","n_windows","R2_share_pos","DA_share_gt_055"]:
            if pd.api.types.is_numeric_dtype(out[c]):
                out[c] = out[c].round(4)
    return out

# -------------------------------------------------------------------
# 1) Tabla por target específico (delta_60, delta_90, ret_60, ret_90)
# -------------------------------------------------------------------
by_target = (
    df.groupby("target", as_index=False)
      .apply(lambda g: summarize(g))
      .reset_index(drop=True)
)

# Score para ordenar: prioriza R2 y DA, penaliza dispersión (R2_std) y MAE alta
by_target["score"] = (
    by_target["R2_mean"].rank(ascending=False, method="min") * 1.0 +
    by_target["DA_mean"].rank(ascending=False, method="min") * 0.7 +
    by_target["R2_std"].rank(ascending=True, method="min") * 0.6 +
    by_target["MAE_mean"].rank(ascending=True, method="min") * 0.3
)
by_target = by_target.sort_values(["score", "target"]).drop(columns=["score"])
by_target = fmt_table(by_target)

print("\n=== 1) Resumen por target (VALID) ===")
display(by_target)

# ------------------------------------------------------
# 2) Tabla por tipo de target: delta vs ret
# ------------------------------------------------------
by_type = (
    df[df["target_type"].isin(["delta","ret"])]
      .groupby("target_type", as_index=False)
      .apply(lambda g: summarize(g))
      .reset_index(drop=True)
      .sort_values("target_type")
)
by_type = fmt_table(by_type)

print("\n=== 2) Resumen por tipo de target: delta vs ret (VALID) ===")
display(by_type)

# -------------------------------------------------------------------
# 3) Comparación por modelo: delta vs ret (promedios por modelo)
# -------------------------------------------------------------------
model_type = (
    df[df["target_type"].isin(["delta","ret"])]
      .groupby(["model","target_type"], as_index=False)
      .apply(lambda g: pd.Series({
          "n_rows": len(g),
          "R2_mean": g["R2"].mean(),
          "DA_mean": g["DA"].mean(),
          "MAE_mean": g["MAE"].mean(),
          "R2_share_pos": (g["R2"] > 0).mean(),
          "DA_share_gt_055": (g["DA"] > 0.55).mean(),
          "R2_std": g["R2"].std(ddof=0),
      }))
      .reset_index(drop=True)
)

# Pivot para diferencias delta - ret (por modelo)
pivot_r2 = model_type.pivot(index="model", columns="target_type", values="R2_mean")
pivot_da = model_type.pivot(index="model", columns="target_type", values="DA_mean")
pivot_mae = model_type.pivot(index="model", columns="target_type", values="MAE_mean")

model_cmp = pd.DataFrame({
    "model": pivot_r2.index,
    "R2_mean_delta": pivot_r2.get("delta"),
    "R2_mean_ret": pivot_r2.get("ret"),
    "R2_delta_minus_ret": pivot_r2.get("delta") - pivot_r2.get("ret"),
    "DA_mean_delta": pivot_da.get("delta"),
    "DA_mean_ret": pivot_da.get("ret"),
    "DA_delta_minus_ret": pivot_da.get("delta") - pivot_da.get("ret"),
    "MAE_mean_delta": pivot_mae.get("delta"),
    "MAE_mean_ret": pivot_mae.get("ret"),
}).reset_index(drop=True)

model_cmp = model_cmp.sort_values("R2_delta_minus_ret", ascending=False)
for c in model_cmp.columns:
    if c != "model" and pd.api.types.is_numeric_dtype(model_cmp[c]):
        model_cmp[c] = model_cmp[c].round(4)

print("\n=== 3) Comparación por modelo: delta vs ret (VALID) ===")
display(model_cmp)

# ------------------------------------------------------
# Decisión sugerida (automática) delta vs ret
# ------------------------------------------------------
# Regla simple: elegir el tipo con:
# - mayor R2_mean
# - mayor DA_mean
# - mayor share R2>0
# - menor R2_std
delta_row = by_type[by_type["target_type"] == "delta"]
ret_row   = by_type[by_type["target_type"] == "ret"]

print("\n=== Decisión sugerida (criterios primarios) ===")
display(by_type)
print("Interpretación recomendada: elija el tipo con mayor R2_mean y DA_mean, mayor R2_share_pos y menor R2_std.")


=== 1) Resumen por target (VALID) ===


,target,n_rows,n_models,n_windows,R2_mean,R2_median,R2_std,R2_share_pos,DA_mean,DA_median,DA_std,DA_share_gt_055,MAE_mean,MAE_median,MAE_std
0,delta_60,40.0,8.0,5.0,-0.0039,-0.0023,0.0039,2.5%,0.4868,0.4796,0.0245,0.0%,37.9025,37.8818,1.7158
3,ret_90,40.0,8.0,5.0,-0.4705,-0.0568,1.2403,0.0%,0.5140,0.5120,0.0261,0.0%,0.0029,0.0027,0.0006
2,ret_60,40.0,8.0,5.0,-0.4269,-0.0231,1.7052,0.0%,0.5126,0.5143,0.0245,0.0%,0.0024,0.0022,0.0010
1,delta_90,40.0,8.0,5.0,-0.0053,-0.0036,0.0050,0.0%,0.4854,0.4773,0.0273,0.0%,47.0327,47.4803,1.6135



=== 2) Resumen por tipo de target: delta vs ret (VALID) ===


,target_type,n_rows,n_models,n_windows,R2_mean,R2_median,R2_std,R2_share_pos,DA_mean,DA_median,DA_std,DA_share_gt_055,MAE_mean,MAE_median,MAE_std
0,delta,80.0,8.0,5.0,-0.0046,-0.0030,0.0046,1.2%,0.4861,0.4784,0.0260,0.0%,42.4676,42.2710,4.8594
1,ret,80.0,8.0,5.0,-0.4487,-0.0293,1.4911,0.0%,0.5133,0.5130,0.0253,0.0%,0.0027,0.0025,0.0009



=== 3) Comparación por modelo: delta vs ret (VALID) ===


,model,R2_mean_delta,R2_mean_ret,R2_delta_minus_ret,DA_mean_delta,DA_mean_ret,DA_delta_minus_ret,MAE_mean_delta,MAE_mean_ret
7,transformer,-0.0103,-1.5928,1.5825,0.4894,0.5131,-0.0237,42.6898,0.0037
3,mlp,-0.0091,-1.4636,1.4545,0.4778,0.5044,-0.0266,42.6589,0.0030
0,gru,-0.0033,-0.3235,0.3201,0.4797,0.5031,-0.0233,42.4521,0.0027
2,lstm,-0.0017,-0.1839,0.1821,0.4747,0.5112,-0.0365,42.3557,0.0026
6,ridge,-0.0084,-0.0240,0.0156,0.4761,0.4734,0.0027,42.7585,0.0024
5,naive_zero,-0.0010,-0.0014,0.0005,NaN,NaN,NaN,42.2499,0.0023
4,naive_mean,-0.0004,-0.0002,-0.0002,0.5440,0.5440,0.0000,42.1884,0.0023
1,lasso,-0.0028,-0.0002,-0.0026,0.4611,0.5440,-0.0829,42.3875,0.0023



=== Decisión sugerida (criterios primarios) ===


,target_type,n_rows,n_models,n_windows,R2_mean,R2_median,R2_std,R2_share_pos,DA_mean,DA_median,DA_std,DA_share_gt_055,MAE_mean,MAE_median,MAE_std
0,delta,80.0,8.0,5.0,-0.0046,-0.0030,0.0046,1.2%,0.4861,0.4784,0.0260,0.0%,42.4676,42.2710,4.8594
1,ret,80.0,8.0,5.0,-0.4487,-0.0293,1.4911,0.0%,0.5133,0.5130,0.0253,0.0%,0.0027,0.0025,0.0009


Interpretación recomendada: elija el tipo con mayor R2_mean y DA_mean, mayor R2_share_pos y menor R2_std.


### **1.1.3. Selección del tipo de target (VALID)**


**1. Comparación delta vs ret**

| Métrica | delta | ret |
|----------|--------|--------|
| R2_mean | -0.0046 | -0.4487 |
| R2_std | 0.0046 | 1.4911 |
| R2_share_pos | 1.2% | 0.0% |
| DA_mean | 0.4861 | 0.5133 |
| DA_share_gt_055 | 0.0% | 0.0% |

---

**2. Análisis estructural**

- **Capacidad explicativa (R²)**

  - Ambos tipos de target presentan **R² promedio negativo**.
  - Sin embargo, la diferencia estructural es muy marcada:

    - **delta:** R2_mean ≈ -0.0046 (muy cercano a 0).
    - **ret:** R2_mean ≈ -0.4487 (muy negativo).

  - La dispersión del desempeño también muestra una diferencia importante:

    - **delta:** R2_std ≈ 0.0046 → comportamiento extremadamente estable.
    - **ret:** R2_std ≈ 1.4911 → desempeño altamente inestable.

  - Aunque delta no logra R² positivo consistente en esta etapa, su desempeño está **mucho más cercano al baseline** que ret.

  **Conclusión:**  
  ret muestra ausencia clara de capacidad explicativa, mientras que delta se mantiene cercano al baseline con baja dispersión.

---

- **Señal direccional (DA)**

  - **delta:** DA_mean ≈ 0.4861
  - **ret:** DA_mean ≈ 0.5133

  Ambos valores se encuentran **muy cercanos al 50%**, lo que indica que **ninguno de los targets presenta señal direccional robusta en esta etapa**.

  Además:

  - Ningún caso supera **DA > 0.55**.
  - Por lo tanto, **no existe evidencia de señal direccional explotable** todavía.

---

- **Robustez entre modelos**

  En la comparación por modelo:

  - En **todos los modelos complejos (Transformer, MLP, GRU, LSTM)** el target **delta presenta R² mucho mayor que ret**.
  - En ret, varios modelos presentan **R² extremadamente negativos** (por ejemplo -1.59 en Transformer y -1.46 en MLP).
  - En delta, todos los modelos se mantienen **muy cerca de R² = 0**, con baja dispersión.

  Esto indica que:

  - **ret introduce alta inestabilidad en el proceso de aprendizaje**.
  - **delta produce un comportamiento mucho más consistente entre arquitecturas.**

---

**3. Conclusión metodológica**

Bajo los criterios definidos:

- Mayor R² promedio.
- Menor dispersión entre modelos.
- Comportamiento más estable entre arquitecturas.

El tipo de target que resulta **metodológicamente más adecuado** es:

> **delta**

Aunque en esta etapa **ninguno de los targets muestra capacidad predictiva fuerte**, el target **ret queda descartado** porque:

- Presenta **R² fuertemente negativo**.
- Muestra **alta inestabilidad entre modelos**.
- Introduce **errores estructurales muy grandes en algunas arquitecturas**.

En cambio, **delta se mantiene cercano al baseline con baja varianza**, lo que lo convierte en un objetivo **más estable para continuar la investigación**.

---

**Próximo paso**

Una vez seleccionado el tipo de target (**delta**), el siguiente análisis consiste en comparar los horizontes:

- **delta_60**
- **delta_90**

con el objetivo de determinar cuál de ellos presenta mejor capacidad predictiva y estabilidad entre modelos.

## **1.2. Selección del horizonte**

Comparar:

- `delta_60`
- `delta_90`

**Criterios**

  - Mayor R² promedio entre modelos.
  - Mejor DA promedio.
  - Menor dispersión entre modelos.

Si `delta_60` domina en estabilidad y consistencia, se elige 60 minutos.  
Si `delta_90` muestra mejor robustez estructural, se elige 90 minutos.


### **1.2.1. Implementación**

In [9]:
import pandas as pd
import numpy as np

# Requiere df_valid con columnas:
# ['model','split','window_size','target','horizon_min','MAE','RMSE','R2','DA']

required = {"model","split","window_size","target","horizon_min","MAE","RMSE","R2","DA"}
missing = required - set(df_valid.columns)
if missing:
    raise ValueError(f"df_valid no tiene columnas requeridas: {sorted(missing)}")

df = df_valid.copy()
df = df[df["split"].astype(str).str.lower().eq("valid")].copy()
df["target"] = df["target"].astype(str).str.lower().str.strip()

# Nos quedamos solo con delta_60 y delta_90
df = df[df["target"].isin(["delta_60", "delta_90"])].copy()

# Flags
df["r2_pos"] = df["R2"] > 0
df["da_gt_055"] = df["DA"] > 0.55

def summarize(g: pd.DataFrame) -> pd.Series:
    return pd.Series({
        "n_rows": len(g),
        "n_models": g["model"].nunique(),
        "n_windows": g["window_size"].nunique(),

        "R2_mean": g["R2"].mean(),
        "R2_median": g["R2"].median(),
        "R2_std": g["R2"].std(ddof=0),
        "R2_share_pos": g["r2_pos"].mean(),

        "DA_mean": g["DA"].mean(),
        "DA_median": g["DA"].median(),
        "DA_std": g["DA"].std(ddof=0),
        "DA_share_gt_055": g["da_gt_055"].mean(),

        "MAE_mean": g["MAE"].mean(),
        "MAE_median": g["MAE"].median(),
        "MAE_std": g["MAE"].std(ddof=0),
    })

def fmt_table(t: pd.DataFrame) -> pd.DataFrame:
    out = t.copy()
    for c in ["R2_share_pos", "DA_share_gt_055"]:
        if c in out.columns:
            out[c] = (out[c] * 100).round(1).astype(str) + "%"
    for c in out.columns:
        if c not in ["target","n_rows","n_models","n_windows","R2_share_pos","DA_share_gt_055"]:
            if pd.api.types.is_numeric_dtype(out[c]):
                out[c] = out[c].round(4)
    return out

# ------------------------------------------------------------
# 1) Resumen agregado por target (delta_60 vs delta_90)
# ------------------------------------------------------------
by_delta = (
    df.groupby("target", as_index=False)
      .apply(lambda g: summarize(g), include_groups=False)
      .reset_index(drop=True)
)

# Score: prioriza R2 y DA; penaliza dispersión (R2_std) y MAE alta
by_delta["score"] = (
    by_delta["R2_mean"].rank(ascending=False, method="min") * 1.0 +
    by_delta["DA_mean"].rank(ascending=False, method="min") * 0.7 +
    by_delta["R2_std"].rank(ascending=True, method="min") * 0.6 +
    by_delta["MAE_mean"].rank(ascending=True, method="min") * 0.3
)

by_delta_sorted = by_delta.sort_values(["score","target"]).drop(columns=["score"])
print("\n=== Resumen delta_60 vs delta_90 (VALID) ===")
display(fmt_table(by_delta_sorted))

# ------------------------------------------------------------
# 2) Robustez por modelo: delta_60 vs delta_90 (promedios por modelo)
# ------------------------------------------------------------
by_model = (
    df.groupby(["model","target"], as_index=False)
      .apply(lambda g: pd.Series({
          "n_rows": len(g),
          "R2_mean": g["R2"].mean(),
          "DA_mean": g["DA"].mean(),
          "MAE_mean": g["MAE"].mean(),
          "R2_std": g["R2"].std(ddof=0),
      }), include_groups=False)
      .reset_index(drop=True)
)

p_r2  = by_model.pivot(index="model", columns="target", values="R2_mean")
p_da  = by_model.pivot(index="model", columns="target", values="DA_mean")
p_mae = by_model.pivot(index="model", columns="target", values="MAE_mean")

cmp = pd.DataFrame({
    "model": p_r2.index,
    "R2_delta_60": p_r2.get("delta_60"),
    "R2_delta_90": p_r2.get("delta_90"),
    "R2_60_minus_90": p_r2.get("delta_60") - p_r2.get("delta_90"),
    "DA_delta_60": p_da.get("delta_60"),
    "DA_delta_90": p_da.get("delta_90"),
    "DA_60_minus_90": p_da.get("delta_60") - p_da.get("delta_90"),
    "MAE_delta_60": p_mae.get("delta_60"),
    "MAE_delta_90": p_mae.get("delta_90"),
}).reset_index(drop=True)

# Para “ganadores” por modelo: cuenta cuántos modelos prefieren 60 vs 90
cmp["winner_r2"] = np.where(cmp["R2_60_minus_90"] > 0, "delta_60",
                     np.where(cmp["R2_60_minus_90"] < 0, "delta_90", "tie"))

win_counts = cmp["winner_r2"].value_counts(dropna=False).rename_axis("winner").reset_index(name="n_models")

# Ordenar la tabla por ventaja en R2
for c in cmp.columns:
    if c not in ["model","winner_r2"]:
        cmp[c] = pd.to_numeric(cmp[c], errors="coerce").round(4)

print("\n=== Comparación por modelo: delta_60 vs delta_90 (VALID) ===")
display(cmp.sort_values("R2_60_minus_90", ascending=False))

print("\n=== Conteo de ganadores por modelo (según R2_mean) ===")
display(win_counts)

# ------------------------------------------------------------
# 3) Decisión automática final (agregada)
# ------------------------------------------------------------
best_target = by_delta.loc[by_delta["score"].idxmin(), "target"]  # score menor = mejor por ranks
print(f"\nDECISIÓN SUGERIDA (por score agregado): {best_target}")


=== Resumen delta_60 vs delta_90 (VALID) ===


,target,n_rows,n_models,n_windows,R2_mean,R2_median,R2_std,R2_share_pos,DA_mean,DA_median,DA_std,DA_share_gt_055,MAE_mean,MAE_median,MAE_std
0,delta_60,40.0,8.0,5.0,-0.0039,-0.0023,0.0039,2.5%,0.4868,0.4796,0.0245,0.0%,37.9025,37.8818,1.7158
1,delta_90,40.0,8.0,5.0,-0.0053,-0.0036,0.0050,0.0%,0.4854,0.4773,0.0273,0.0%,47.0327,47.4803,1.6135



=== Comparación por modelo: delta_60 vs delta_90 (VALID) ===


,model,R2_delta_60,R2_delta_90,R2_60_minus_90,DA_delta_60,DA_delta_90,DA_60_minus_90,MAE_delta_60,MAE_delta_90,winner_r2
3,mlp,-0.0070,-0.0113,0.0043,0.4803,0.4754,0.0049,38.0326,47.2852,delta_60
7,transformer,-0.0088,-0.0117,0.0029,0.4878,0.4910,-0.0033,38.0830,47.2965,delta_60
6,ridge,-0.0076,-0.0093,0.0017,0.4751,0.4770,-0.0020,38.1300,47.3869,delta_60
1,lasso,-0.0020,-0.0036,0.0017,0.4666,0.4556,0.0110,37.7905,46.9845,delta_60
0,gru,-0.0031,-0.0036,0.0004,0.4785,0.4810,-0.0024,37.9223,46.9820,delta_60
5,naive_zero,-0.0009,-0.0011,0.0002,NaN,NaN,NaN,37.7350,46.7648,delta_60
2,lstm,-0.0017,-0.0018,0.0001,0.4764,0.4729,0.0035,37.8397,46.8716,delta_60
4,naive_mean,-0.0003,-0.0004,0.0001,0.5428,0.5451,-0.0023,37.6864,46.6903,delta_60



=== Conteo de ganadores por modelo (según R2_mean) ===


,winner,n_models
0,delta_60,8



DECISIÓN SUGERIDA (por score agregado): delta_60


### **1.2.2. Selección del horizonte dentro de delta (VALID)**

**1. Ventaja consistente de delta_60**

- A nivel agregado

  - **R2_mean**
    - delta_60 = -0.0039
    - delta_90 = -0.0053  
    delta_60 presenta un R² promedio ligeramente superior (más cercano a 0).

  - **DA_mean**
    - delta_60 = 0.4868
    - delta_90 = 0.4854  
    La señal direccional es prácticamente equivalente, con una ligera ventaja para delta_60.

  - **MAE_mean**
    - delta_60 = 37.90
    - delta_90 = 47.03  
    La diferencia es significativa: el error absoluto promedio es **mucho menor en delta_60**.

  - **R2_share_pos**
    - delta_60 = 2.5%
    - delta_90 = 0.0%

  **Conclusión:**  
  Aunque ninguno de los horizontes muestra capacidad predictiva fuerte en esta etapa, **delta_60 domina consistentemente en todas las métricas principales**.

---

**2. Consistencia por modelo**

Todos los modelos favorecen **delta_60**.

Resultados:

- **8 de 8 modelos** prefieren delta_60 según R² promedio.
- **Ningún modelo** muestra mejor desempeño con delta_90.
- **No existen empates.**

Esto indica que la ventaja de delta_60 **no depende de una arquitectura específica**.

La mejora aparece en:

- modelos deep learning (Transformer, MLP, GRU, LSTM),
- modelos lineales (Ridge, Lasso),
- baselines (Naive).

---

**3. Estabilidad**

- **delta_60**
  - R2_std = 0.0039

- **delta_90**
  - R2_std = 0.0050

delta_60 presenta **menor dispersión del desempeño**, lo que indica mayor estabilidad entre configuraciones.

Por lo tanto:

- delta_60 no solo obtiene mejores métricas promedio,
- también muestra **comportamiento más estable entre modelos y ventanas**.

---

**4. Interpretación estructural**

En el dataset intradía del MNQ:

- El horizonte de **60 minutos** parece alinearse mejor con la dinámica intradía del mercado.
- Al extender el horizonte a **90 minutos**:

  - aumenta la acumulación de ruido,
  - se reduce la relación señal/ruido,
  - los errores de predicción crecen significativamente.

Este comportamiento es consistente con muchos problemas de predicción intradía, donde horizontes más largos tienden a diluir la señal disponible en las features de corto plazo.

---

**5. Implicación para el pipeline**

La decisión es clara:

- **Tipo de target seleccionado:** delta  
- **Horizonte seleccionado:** **delta_60**

El horizonte de 90 minutos queda descartado en esta etapa debido a:

- mayor error absoluto,
- menor R² promedio,
- mayor dispersión del desempeño.

---

**6. Nivel de confianza**

La decisión tiene un **nivel de confianza alto** porque:

- delta_60 supera a delta_90 en todas las métricas relevantes.
- La ventaja se observa **en todos los modelos evaluados (8/8)**.
- delta_60 presenta **menor dispersión (R2_std)**.
- El **score agregado del análisis confirma la selección automática**.

En consecuencia, el proceso de selección del horizonte puede considerarse **metodológicamente cerrado**, adoptando **delta_60 como target final para el pipeline de modelado**.

## **1.3. Selección de windows_size**

Una vez fijado el target y horizonte: `delta_60`

Para cada `window_size` evaluar:

- Promedio de R² entre modelos.
- Mejor R² alcanzado.
- Consistencia (por ejemplo, cantidad de modelos con R² > 0.25).

**Criterio recomendado**

- No elegir la ventana únicamente por el mejor modelo individual.
- Elegir la ventana con mejor desempeño agregado y estabilidad.

Esto reduce el riesgo de sobreajuste estructural.

### **1.3.1. Implementación**

In [10]:
import pandas as pd
import numpy as np

# Filtrado base
df = df_valid.copy()
df = df[df["split"].str.lower() == "valid"]
df = df[df["target"].str.lower() == "delta_60"].copy()

# ------------------------------------------------------------
# 1) Resumen agregado por window_size (criterio reforzado)
# ------------------------------------------------------------

summary = (
    df.groupby("window_size")
      .agg(
          n_rows=("R2", "size"),
          n_models=("model", "nunique"),

          # Performance central
          R2_mean=("R2", "mean"),
          R2_median=("R2", "median"),
          R2_std=("R2", lambda x: x.std(ddof=0)),
          R2_max=("R2", "max"),

          # Consistencia fuerte
          n_models_R2_gt_025=("R2", lambda x: (x > 0.25).sum()),

          DA_mean=("DA", "mean"),
          MAE_mean=("MAE", "mean"),
      )
      .reset_index()
)

summary = summary.sort_values("R2_mean", ascending=False)

print("\n=== Resumen reforzado por window_size (delta_60 | VALID) ===")
display(summary.round(4))

# ------------------------------------------------------------
# 2) Ranking estructural balanceado
# ------------------------------------------------------------

summary["score_balanceado"] = (
    summary["R2_mean"].rank(ascending=False) * 1.0 +
    summary["R2_std"].rank(ascending=True) * 0.8 +
    summary["n_models_R2_gt_025"].rank(ascending=False) * 0.7 +
    summary["R2_max"].rank(ascending=False) * 0.5
)

ranking = summary.sort_values("score_balanceado").drop(columns=["score_balanceado"])

print("\n=== Ranking estructural balanceado ===")
display(ranking.round(4))

# ------------------------------------------------------------
# 3) Ganadores por modelo (sin sesgo individual)
# ------------------------------------------------------------

by_model = (
    df.groupby(["model","window_size"])
      .agg(R2_mean=("R2","mean"))
      .reset_index()
)

idx = by_model.groupby("model")["R2_mean"].idxmax()
best_per_model = by_model.loc[idx]

win_counts = (
    best_per_model["window_size"]
    .value_counts()
    .rename_axis("window_size")
    .reset_index(name="n_models_ganan")
    .sort_values("window_size")
)

print("\n=== Conteo de ganadores por modelo ===")
display(win_counts)


=== Resumen reforzado por window_size (delta_60 | VALID) ===


,window_size,n_rows,n_models,R2_mean,R2_median,R2_std,R2_max,n_models_R2_gt_025,DA_mean,MAE_mean
2,90,8,8,-0.0032,-0.0020,0.0029,-0.0004,0,0.4903,37.9183
0,30,8,8,-0.0035,-0.0038,0.0022,-0.0005,0,0.4855,35.4501
1,60,8,8,-0.0038,-0.0027,0.0027,-0.0005,0,0.4818,36.6608
4,180,8,8,-0.0042,-0.0015,0.0051,0.0007,0,0.4882,40.0976
3,120,8,8,-0.0048,-0.0029,0.0054,-0.0002,0,0.4880,39.3855



=== Ranking estructural balanceado ===


,window_size,n_rows,n_models,R2_mean,R2_median,R2_std,R2_max,n_models_R2_gt_025,DA_mean,MAE_mean
2,90,8,8,-0.0032,-0.0020,0.0029,-0.0004,0,0.4903,37.9183
0,30,8,8,-0.0035,-0.0038,0.0022,-0.0005,0,0.4855,35.4501
1,60,8,8,-0.0038,-0.0027,0.0027,-0.0005,0,0.4818,36.6608
4,180,8,8,-0.0042,-0.0015,0.0051,0.0007,0,0.4882,40.0976
3,120,8,8,-0.0048,-0.0029,0.0054,-0.0002,0,0.4880,39.3855



=== Conteo de ganadores por modelo ===


,window_size,n_models_ganan
2,90,1
1,120,1
0,180,6


### **1.3.2. Selección de window_size bajo criterio estadístico y criterio profesional de trading**

**1. Lectura objetiva de los resultados**

- **R² promedio (calidad media)**

| window | R2_mean |
|--------|---------|
| 90  | -0.0032 |
| 30  | -0.0035 |
| 60  | -0.0038 |
| 180 | -0.0042 |
| 120 | -0.0048 |

  - **90** presenta el mejor R² promedio (más cercano a 0).
  - **30 y 60** se ubican muy cerca.
  - **180 y 120** muestran desempeño inferior.

---

- **Techo potencial (R2_max)**

| window | R2_max |
|--------|--------|
| 180 | 0.0007 |
| 90  | -0.0004 |
| 30  | -0.0005 |
| 60  | -0.0005 |
| 120 | -0.0002 |

  - **180** presenta el mayor R² máximo observado.
  - Sin embargo, el valor es extremadamente cercano a cero, lo que indica que **ninguna ventana logra capacidad explicativa fuerte en esta etapa**.

---

- **Consistencia fuerte (R² > 0.25)**

| window | modelos > 0.25 |
|--------|----------------|
| 30  | 0 |
| 60  | 0 |
| 90  | 0 |
| 120 | 0 |
| 180 | 0 |

  Ninguna configuración alcanza **R² > 0.25**, lo cual confirma que el problema en esta etapa se mantiene **muy cercano al baseline**.

---

- **Estabilidad (R2_std)**

| window | R2_std |
|--------|--------|
| 30  | 0.0022 |
| 60  | 0.0027 |
| 90  | 0.0029 |
| 180 | 0.0051 |
| 120 | 0.0054 |

  - **30** es la ventana más estable.
  - **90** mantiene buena estabilidad.
  - **180 y 120** muestran mayor dispersión.

---

- **Señal direccional (DA_mean)**

| window | DA_mean |
|--------|---------|
| 90  | 0.4903 |
| 180 | 0.4882 |
| 120 | 0.4880 |
| 30  | 0.4855 |
| 60  | 0.4818 |

  - **90** presenta la mayor señal direccional promedio.
  - Las diferencias entre ventanas son pequeñas.

---

**2. Ganadores por modelo**

| window | modelos que lo prefieren |
|--------|--------------------------|
| 180 | 6 |
| 90  | 1 |
| 120 | 1 |

Esto indica que:

- **180 es la ventana que más veces produce el mejor modelo individual.**
- Sin embargo, el criterio metodológico definido **no prioriza el mejor modelo individual**, sino el desempeño agregado.

---

**3. Interpretación estratégica (criterio estadístico)**

Analizando las métricas agregadas:

- **90**
  - Mejor R² promedio.
  - Mejor señal direccional.
  - Estabilidad aceptable.

- **30**
  - Máxima estabilidad.
  - R² ligeramente inferior.

- **180**
  - Mayor techo potencial.
  - Mayor frecuencia de ganadores por modelo.
  - Mayor dispersión.

Perfiles:

- **90** → mejor desempeño agregado.
- **30** → perfil más estable.
- **180** → mayor potencial individual pero mayor varianza.

---

**4. Enfoque metodológico del experimento**

El criterio definido para la selección de ventana es:

> elegir la ventana con **mejor desempeño agregado y estabilidad**, no la que produzca el mejor modelo individual.

Bajo este criterio:

- El hecho de que **180 gane en 6 modelos** no es suficiente si su desempeño promedio es inferior y más disperso.

---

**5. Conclusión final**

Considerando:

- mejor **R² promedio**,
- mejor **DA promedio**,
- estabilidad razonable,
- y buen equilibrio entre métricas,

la ventana que presenta **mejor desempeño estructural agregado** es:

> **window_size = 90**

Esta ventana ofrece el **mejor balance entre capacidad predictiva promedio y estabilidad**, evitando depender de un modelo individual excepcional.

---

**Decisión adoptada**

Se selecciona:

> **window_size = 90**

como configuración de ventana para el pipeline final con target **delta_60**.

## **1.4. Selección de modelos sobre objetivo**

Una vez definidos:

- target: delta
- horizon: delta_60
- window_size: 90

Ordenar los modelos por:

- R² (criterio principal)
- MAE (criterio secundario)
- DA (validación direccional)

Seleccionar:

- El mejor modelo absoluto.
- El segundo mejor modelo que sea estructuralmente diferente.


### **1.4.1. Implementación**


In [11]:
import pandas as pd
import numpy as np

# Requiere df_valid (o df_seq2one_all). Ajusta el nombre acá:
df = df_valid.copy()

# -------------------------------------------------
# 1) Filtrar: VALID + delta_60 + window_size=90
# -------------------------------------------------
df_f = df[
    (df["split"].astype(str).str.lower() == "valid") &
    (df["target"].astype(str).str.lower() == "delta_60") &
    (df["window_size"].astype(int) == 90)
].copy()

print("Rows:", len(df_f))
print("Models:", sorted(df_f["model"].unique()))

# -------------------------------------------------
# 2) Resumen por modelo (una fila por modelo)
# -------------------------------------------------
summary_model = (
    df_f.groupby("model")
        .agg(
            n_rows=("R2", "size"),
            MAE_mean=("MAE", "mean"),
            MAE_median=("MAE", "median"),
            RMSE_mean=("RMSE", "mean"),
            R2_mean=("R2", "mean"),
            R2_median=("R2", "median"),
            DA_mean=("DA", "mean"),
            DA_median=("DA", "median"),
        )
        .reset_index()
)

# Ranking según criterio: R2 desc, MAE asc, DA desc
summary_model = summary_model.sort_values(
    ["R2_mean", "MAE_mean", "DA_mean"],
    ascending=[False, True, False]
)

print("\n=== MODEL RANKING (VALID | delta_60 | L=90) ===")
display(summary_model.round(4))

# -------------------------------------------------
# 3) Top combinaciones puntuales (por si hay varias filas por modelo)
#    Orden: R2 desc, MAE asc, DA desc
# -------------------------------------------------
top_rows = (
    df_f.sort_values(["R2", "MAE", "DA"], ascending=[False, True, False])
        .head(10)
)

print("\n=== TOP 10 ROWS (VALID | delta_60 | L=90) ===")
display(top_rows[["model","window_size","MAE","RMSE","R2","DA"]].round(4))

# -------------------------------------------------
# 4) Selección sugerida de 2 modelos:
#    - mejor absoluto (top 1)
#    - segundo "estructuralmente diferente" (heurística por familia)
# -------------------------------------------------

# Heurística simple de familias (ajústala si tus nombres cambian)
def model_family(m: str) -> str:
    m = str(m).lower()
    if m in {"ridge","lasso","linear","elasticnet"}:
        return "linear"
    if m in {"rf","random_forest","xgb","xgboost","lgbm","lightgbm","catboost","gbm"}:
        return "tree_ensemble"
    if m in {"mlp"}:
        return "mlp"
    if m in {"lstm","gru","rnn","tcn"}:
        return "sequence"
    if "transformer" in m or m in {"tft"}:
        return "attention"
    return "other"

summary_model["family"] = summary_model["model"].apply(model_family)

best_model = summary_model.iloc[0][["model","family","R2_mean","MAE_mean","DA_mean"]].to_dict()

# Buscar el mejor modelo que NO sea de la misma familia
best_family = best_model["family"]
candidates = summary_model[summary_model["family"] != best_family].copy()

second_model = None
if len(candidates) > 0:
    second_model = candidates.iloc[0][["model","family","R2_mean","MAE_mean","DA_mean"]].to_dict()

print("\n=== SELECCIÓN SUGERIDA ===")
print("1) Mejor absoluto:", best_model)
print("2) Segundo (familia distinta):", second_model)


Rows: 8
Models: ['gru', 'lasso', 'lstm', 'mlp', 'naive_mean', 'naive_zero', 'ridge', 'transformer']

=== MODEL RANKING (VALID | delta_60 | L=90) ===


,model,n_rows,MAE_mean,MAE_median,RMSE_mean,R2_mean,R2_median,DA_mean,DA_median
4,naive_mean,1,37.7366,37.7366,52.8225,-0.0004,-0.0004,0.5414,0.5414
5,naive_zero,1,37.7744,37.7744,52.8341,-0.0008,-0.0008,NaN,NaN
7,transformer,1,37.8694,37.8694,52.8537,-0.0016,-0.0016,0.5040,0.5040
2,lstm,1,37.8943,37.8943,52.8639,-0.0019,-0.0019,0.4738,0.4738
1,lasso,1,37.8403,37.8403,52.8665,-0.0020,-0.0020,0.4625,0.4625
0,gru,1,37.9342,37.9342,52.8892,-0.0029,-0.0029,0.4847,0.4847
3,mlp,1,38.0924,38.0924,53.0204,-0.0079,-0.0079,0.4883,0.4883
6,ridge,1,38.2046,38.2046,53.0325,-0.0083,-0.0083,0.4769,0.4769



=== TOP 10 ROWS (VALID | delta_60 | L=90) ===


,model,window_size,MAE,RMSE,R2,DA
177,naive_mean,90,37.7366,52.8225,-0.0004,0.5414
217,naive_zero,90,37.7744,52.8341,-0.0008,NaN
297,transformer,90,37.8694,52.8537,-0.0016,0.5040
97,lstm,90,37.8943,52.8639,-0.0019,0.4738
57,lasso,90,37.8403,52.8665,-0.0020,0.4625
17,gru,90,37.9342,52.8892,-0.0029,0.4847
137,mlp,90,38.0924,53.0204,-0.0079,0.4883
257,ridge,90,38.2046,53.0325,-0.0083,0.4769



=== SELECCIÓN SUGERIDA ===
1) Mejor absoluto: {'model': 'naive_mean', 'family': 'other', 'R2_mean': -0.00036919355816622534, 'MAE_mean': 37.736593323945044, 'DA_mean': 0.541398943589509}
2) Segundo (familia distinta): {'model': 'transformer', 'family': 'attention', 'R2_mean': -0.0015522769254341373, 'MAE_mean': 37.8694265611462, 'DA_mean': 0.5040472869457846}


### **1.4.2. Selección de modelos (VALID | delta_60 | window_size = 90)**


**1. Ranking de modelos (excluyendo baseline)**

Los modelos *naive* se utilizan únicamente como referencia de desempeño y no se consideran candidatos para el modelo final.

Por lo tanto, el ranking se analiza únicamente sobre los modelos que realmente aprenden patrones.

| Modelo | R2_mean | MAE_mean | DA_mean |
|------|------|------|------|
| transformer | -0.0016 | 37.8694 | 0.5040 |
| lstm | -0.0019 | 37.8943 | 0.4738 |
| lasso | -0.0020 | 37.8403 | 0.4625 |
| gru | -0.0029 | 37.9342 | 0.4847 |
| mlp | -0.0079 | 38.0924 | 0.4883 |
| ridge | -0.0083 | 38.2046 | 0.4769 |

---

**2. Mejor modelo absoluto**

El modelo con mejor desempeño global entre los modelos reales es:

> **Transformer**

Porque:

- Tiene el **mejor R² promedio** entre los modelos que aprenden.
- Mantiene **MAE competitivo**.
- Presenta **DA cercana a 0.5**, consistente con la dificultad del problema.

---

**3. Segundo modelo estructuralmente diferente**

Siguiendo el criterio metodológico de seleccionar un segundo modelo de **familia diferente**, el siguiente candidato es:

> **LSTM**

Familias:

| Modelo | Familia |
|------|------|
| Transformer | Attention |
| LSTM | Sequence model |

Esto permite evaluar dos paradigmas distintos de modelado de series temporales:

- **Transformer** → modelos basados en atención.
- **LSTM** → modelos recurrentes clásicos.

---

**4. Interpretación estructural**

Los resultados indican que:

- Ningún modelo logra todavía superar claramente al baseline naive.
- Sin embargo, los modelos de **deep learning secuencial** (Transformer, LSTM) se posicionan consistentemente en los primeros lugares.

Esto sugiere que la señal predictiva, aunque débil, probablemente esté asociada a **patrones temporales complejos** que estos modelos pueden capturar mejor que modelos lineales o MLP.

---

**5. Decisión metodológica**

Para continuar el desarrollo del pipeline se seleccionan:

**Modelo principal**

> **Transformer**

**Modelo de comparación estructural**

> **LSTM**

Esta elección permite:

- evaluar dos arquitecturas diferentes,
- evitar depender de una sola familia de modelos,
- reducir el riesgo de sesgo arquitectural.

---

**Configuración final del pipeline**

| Componente | Selección |
|------|------|
| Target | delta |
| Horizonte | delta_60 |
| Window size | 90 |
| Modelo principal | Transformer |
| Modelo alternativo | LSTM |

## **1.5. Análisis de Transformer**

In [15]:
df_transformer = df_seq2one_all[
    df_seq2one_all["model"].astype(str).str.lower().str.contains("transformer")
].copy()

print("Rows:", len(df_transformer))
display(df_transformer)

Rows: 40


,model,split,window_size,target,horizon_min,MAE,RMSE,R2,DA
280,transformer,test,30,delta_60,60,53.925824,83.363054,-0.001194,0.473361
281,transformer,valid,30,delta_60,60,35.489670,50.321206,-0.004466,0.494049
282,transformer,test,30,delta_90,90,68.869327,104.479113,-0.009455,0.474374
283,transformer,valid,30,delta_90,90,45.277670,63.533336,-0.022286,0.477252
284,transformer,test,30,ret_60,60,0.003270,0.005726,-0.788453,0.507247
285,transformer,valid,30,ret_60,60,0.002312,0.003181,-0.312625,0.514330
286,transformer,test,30,ret_90,90,0.003605,0.007266,-0.843234,0.494261
287,transformer,valid,30,ret_90,90,0.002602,0.003616,-0.083587,0.524706
288,transformer,test,60,delta_60,60,55.493872,85.245774,0.000877,0.475550
289,transformer,valid,60,delta_60,60,36.866469,51.640016,-0.007466,0.472685


In [16]:
df_v = df_transformer[df_transformer["split"] == "valid"].copy()

summary_target = (
    df_v.groupby("target")
    .agg(
        R2_mean=("R2","mean"),
        R2_max=("R2","max"),
        MAE_mean=("MAE","mean"),
        DA_mean=("DA","mean")
    )
    .sort_values("R2_mean", ascending=False)
)

display(summary_target.round(4))

,R2_mean,R2_max,MAE_mean,DA_mean
target,,,,
delta_60,-0.0088,-0.0016,38.0830,0.4878
delta_90,-0.0117,-0.0076,47.2965,0.4910
ret_90,-0.6271,-0.0836,0.0035,0.5185
ret_60,-2.5584,-0.2885,0.0038,0.5076


In [17]:
df_d60 = df_v[df_v["target"] == "delta_60"]

summary_window = (
    df_d60.groupby("window_size")
    .agg(
        R2=("R2","mean"),
        MAE=("MAE","mean"),
        DA=("DA","mean")
    )
    .sort_values("R2", ascending=False)
)

display(summary_window.round(4))

,R2,MAE,DA
window_size,,,
90,-0.0016,37.8694,0.5040
30,-0.0045,35.4897,0.4940
60,-0.0075,36.8665,0.4727
180,-0.0128,40.3953,0.4847
120,-0.0178,39.7943,0.4833


In [18]:
df_test = df_transformer[
    (df_transformer["split"]=="test") &
    (df_transformer["target"]=="delta_60") &
    (df_transformer["window_size"]==90)
]

display(df_test.round(4))

,model,split,window_size,target,horizon_min,MAE,RMSE,R2,DA
296,transformer,test,90,delta_60,60,56.8983,87.2457,0.0004,0.4887


## **1.6. Análisis de LSTM**

In [23]:
df_lstm = df_seq2one_all[
    df_seq2one_all["model"].astype(str).str.lower().str.contains("lstm")
].copy()

#print("Rows:", len(df_lstm))
#display(df_lstm)

df_v = df_lstm[df_lstm["split"] == "valid"].copy()

summary_target = (
    df_v.groupby("target")
    .agg(
        R2_mean=("R2","mean"),
        R2_max=("R2","max"),
        MAE_mean=("MAE","mean"),
        DA_mean=("DA","mean")
    )
    .sort_values("R2_mean", ascending=False)
)

display(summary_target.round(4))

df_d60 = df_v[df_v["target"] == "delta_60"]

summary_window = (
    df_d60.groupby("window_size")
    .agg(
        R2=("R2","mean"),
        MAE=("MAE","mean"),
        DA=("DA","mean")
    )
    .sort_values("R2", ascending=False)
)

display(summary_window.round(4))

df_test = df_lstm[
    (df_lstm["split"]=="test") &
    (df_lstm["target"]=="delta_60") &
    (df_lstm["window_size"]==180)
]

display(df_test.round(4))

,R2_mean,R2_max,MAE_mean,DA_mean
target,,,,
delta_60,-0.0017,0.0007,37.8397,0.4764
delta_90,-0.0018,-0.0005,46.8716,0.4729
ret_60,-0.1721,-0.1483,0.0023,0.5094
ret_90,-0.1956,-0.1460,0.0029,0.5130


,R2,MAE,DA
window_size,,,
180,0.0007,39.9498,0.4811
120,-0.0017,39.2990,0.4807
90,-0.0019,37.8943,0.4738
60,-0.0022,36.6065,0.4719
30,-0.0031,35.4492,0.4743


,model,split,window_size,target,horizon_min,MAE,RMSE,R2,DA
112,lstm,test,180,delta_60,60,61.6684,93.1437,0.0002,0.4791


# **2. SEQ2ONE - Testeo de robustez**

## **2.1. Conclusión final del análisis comparativo L=90 vs L=180 (Transformer – delta_60)**

El análisis realizado es metodológicamente completo y robusto:

- Se evaluaron 20 seeds independientes.
- Se realizó comparación descriptiva por ventana y split.
- Se aplicaron tests estadísticos formales (t-test pareado y Wilcoxon).
- Se calculó tamaño del efecto (Cohen’s d).
- Se analizó la distribución de diferencias por seed.
- Se evaluó la consistencia de la mejora (75% de las seeds mejoran).
- Se revisó el gap de generalización (valid vs test).

La conclusión no se basa en una observación puntual, sino en evidencia estadística y estructural consistente.

Conclusión técnica:

- La ventana L=180 es superior a L=90 para el target `delta_60`.
- La mejora es estadísticamente significativa (p < 0.05).
- El tamaño del efecto es moderado (d ≈ 0.57).
- La mejora es consistente en la mayoría de inicializaciones.
- No se observa incremento del overfitting.
- La mejora no depende de una única seed extrema.

En consecuencia, el análisis puede considerarse formalmente cerrado y la ventana L=180 puede adoptarse como configuración preferente para el Transformer en este target.

## **2.2. Conclusión final del análisis comparativo L=90 vs L=180 (MLP – delta_60)**

El análisis realizado es metodológicamente completo y robusto:

- Se evaluaron 20 seeds independientes.
- Se realizó comparación descriptiva por ventana y split.
- Se aplicaron tests estadísticos formales (t-test pareado y Wilcoxon).
- Se calculó tamaño del efecto (Cohen’s d).
- Se analizó la distribución de diferencias por seed.
- Se evaluó la consistencia de la mejora (95% de las seeds mejoran).
- Se revisó el gap de generalización (valid vs test).
- La conclusión se basa en evidencia estadística y estructural consistente, no en observaciones puntuales.

Conclusión técnica:

La ventana L=180 es superior a L=90 para el target delta_60 en el MLP.

- La mejora es altamente significativa (p << 0.05).
- El tamaño del efecto es muy grande (d ≈ 1.84).
- La mejora es consistente en prácticamente todas las inicializaciones (19 de 20).
- El gap VALID–TEST se reduce notablemente.
- No se observa incremento del overfitting.
- El deterioro en el peor caso es pequeño y no altera la conclusión global.
- La mejora no depende de seeds extremas ni de outliers.

En consecuencia, el análisis puede considerarse formalmente cerrado y la ventana L=180 puede adoptarse como configuración preferente para el MLP en este target.


## **2.3. Revisión y actualización de la selección de window_size tras validación estadística multi-seed**

En el punto **1.3.2. Selección de window_size bajo criterio estadístico y criterio profesional de trading** se decidió tomar como mejor `window_size` el tamaño 90, pero tras una investigación rigurosa y metodológicamente más sólida, dicha decisión fue revisada.

La selección inicial de `window_size = 90` se basó en:

- Comparación agregada entre múltiples modelos.
- Métricas promedio (R²_mean, DA_mean).
- Evaluación de estabilidad relativa (R²_std).
- Criterio profesional de trading orientado a robustez operativa.
- Ausencia de análisis multi-seed con modelo e hiperparámetros fijos.

En ese contexto, 90 representaba un equilibrio razonable entre rendimiento promedio, estabilidad y señal direccional. Sin embargo, esa era una decisión exploratoria de carácter transversal (multi-modelo), no un contraste estadístico formal intra-modelo.

Posteriormente se realizó un análisis metodológicamente superior que incluyó:

- Fijación estricta de modelo e hiperparámetros.
- Evaluación con 20 seeds independientes.
- Aplicación de tests estadísticos formales (paired t-test y Wilcoxon).
- Cálculo de tamaño del efecto (Cohen’s d).
- Análisis de consistencia de mejora por seed.
- Evaluación del gap de generalización (VALID vs TEST).

Este nuevo análisis cambia sustancialmente el nivel de evidencia disponible.

Resultados clave del análisis intra-modelo:

- En **Transformer**, L=180 es estadísticamente superior a L=90.
- En **MLP**, L=180 es claramente y fuertemente superior.
- La mejora es consistente en la mayoría (Transformer) y casi totalidad (MLP) de las seeds.
- No se observa incremento del overfitting.
- La mejora no depende de seeds extremas ni de outliers.

Por lo tanto, no existe una contradicción con la decisión anterior, sino un **refinamiento basado en mayor evidencia estadística**.

El primer análisis respondía a la pregunta:
> ¿Qué ventana parece más equilibrada en un análisis exploratorio general?

El segundo responde:
> Dado un modelo fijo, ¿existe evidencia estadística robusta de superioridad?

La respuesta empírica es afirmativa: **L=180 domina a L=90** bajo contraste formal.

Desde un criterio riguroso orientado a maximizar poder predictivo real con evidencia estadística replicable, la decisión racional es:

**Adoptar window_size = 180 para el target delta_60 en MLP y Transformer.**

La ventana 90 podría seguir siendo defendible únicamente bajo criterios operativos específicos (menor complejidad computacional, menor latencia, mayor reactividad a cambios de régimen extremos). No obstante, desde el punto de vista estadístico y estructural del modelo, la evidencia actual favorece consistentemente a 180.

El proceso seguido —exploración → hipótesis → test formal → confirmación— refleja una evolución metodológica correcta.

En consecuencia, la decisión anterior queda superada por evidencia más profunda y formal, y la configuración preferente pasa a ser:

**window_size = 180.**

**Dado que delta_60 domina estructuralmente en todos los modelos y métricas, no se considera necesario un análisis multi-seed adicional para confirmar esta decisión.**

# **3. SEQ20NE - Tuneo de HP**

**Pregunta metodológica**

> ¿Conviene usar una única seed fija durante todo el proceso de tuning?

Respuesta corta:  
- No conviene utilizar una seed arbitraria y única como criterio definitivo de selección.

---

**1. Qué ocurre si se usa una sola seed fija**

Si el tuning se realiza con una única inicialización fija, por ejemplo:

```python
torch.manual_seed(42)
```

y se mantiene constante durante todo el proceso:

- Se optimizan los hiperparámetros para una trayectoria específica de inicialización.
- El óptimo encontrado puede depender parcialmente del azar.
- Puede seleccionarse una configuración que funciona bien únicamente bajo esa seed.

Esto introduce un sesgo oculto por inicialización.

---

**2. Estrategias estándar en práctica rigurosa**

Existen tres enfoques habituales:

**Opción A — Seed fija única (simple pero débil)**

- Usar una seed fija (por ejemplo, 42).
- Comparar hiperparámetros bajo exactamente la misma inicialización.
- Ventaja:
  - Comparación limpia y controlada entre configuraciones.
- Desventaja:
  - Riesgo de sobreajuste a la seed.
  - No evalúa robustez.
  
  Aceptable para exploración rápida, no ideal para cierre formal.

**Opción B — Seed fija durante tuning + multi-seed en validación final (recomendado)**

Proceso:

1. Durante el tuning:
    - Usar una seed fija (ej. 42).
    - Buscar las mejores configuraciones según VALID.

2. Después del tuning:
    - Tomar la mejor (o top 3) configuraciones.
    - Evaluarlas con 10–20 seeds independientes.
    - Reportar media, desviación estándar y, si corresponde, tests estadísticos.

Ventajas:
- El tuning es computacionalmente eficiente.
- La robustez se evalúa de forma explícita al final.
- Se evita explosión combinatoria.

Este es el enfoque más equilibrado entre rigor y costo computacional.

**Opción C — Multi-seed dentro del tuning (ultra riguroso)**

Para cada combinación de hiperparámetros:
- Ejecutar múltiples seeds (ej. 5).
- Optimizar sobre el promedio.

Ventajas:
- Máximo rigor estadístico.

Desventajas:
- Muy costoso computacionalmente.
- Poco práctico en deep learning intradía.

Recomendable solo si el problema es extremadamente sensible a la inicialización.

---

**3. Estrategia adoptada para este proyecto**

Dado el nivel de rigurosidad actual del pipeline:

Durante tuning (Optuna / grid search):
- Usar seed fija = 42.
- Comparar configuraciones en VALID.
- Seleccionar top 1 o top 3.

Después del tuning:
- Ejecutar las configuraciones seleccionadas con 20 seeds.
- Elegir la que:
  - Maximice R² medio.
  - Minimice R²_std.
  - Mantenga gap VALID–TEST controlado.

Esto mantiene coherencia con el enfoque multi-seed ya aplicado en la selección de `window_size`.

---

**4. Qué no debe hacerse**

- No seleccionar “la mejor seed”.
- No reportar únicamente la mejor corrida.
- No variar la seed hasta encontrar un resultado favorable.

Eso constituiría data snooping por inicialización.

---


**5. Detalle técnico para control de reproducibilidad**

Si se utiliza PyTorch (con CUDA y Dropout), fijar explícitamente:

```python
torch.manual_seed(seed)
np.random.seed(seed)
random.seed(seed)
torch.cuda.manual_seed_all(seed)
```

Para determinismo más estricto:

```python
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
```

(Esto puede reducir ligeramente el rendimiento.)

---

**6. Conclusión**

Para este proyecto:
- Tuning → seed fija (ej. 42).
- Validación final del mejor modelo → 20 seeds.
- Reportar media, std y contraste estadístico.

Este procedimiento garantiza rigor metodológico sin multiplicar innecesariamente el costo computacional.

## **3.1. Tuneo de Transfomer**



| cfg_id | R2_valid_mean | R2_valid_std | R2_test_mean | R2_test_std | DA_valid_mean | DA_test_mean | MAE_valid_mean | MAE_test_mean | gap_valid_minus_test_mean |
|-------:|--------------:|-------------:|-------------:|------------:|--------------:|-------------:|---------------:|--------------:|---------------------------:|
| 2 | 0.586100 | 0.009491 | 0.476808 | 0.021605 | 0.791358 | 0.784349 | 18.050566 | 30.365389 | 0.109291 |
| 3 | 0.586208 | 0.008550 | 0.475199 | 0.020677 | 0.791879 | 0.783489 | 18.020130 | 30.380457 | 0.111010 |
| 1 | 0.582239 | 0.009785 | 0.461825 | 0.025825 | 0.787274 | 0.781453 | 18.306551 | 30.866381 | 0.120414 |


1. **La configuración 2 presenta el mejor equilibrio entre rendimiento y estabilidad**

   En promedio, alcanza el mayor `R2_test_mean` y mantiene una dispersión controlada entre seeds, lo que indica que su desempeño no depende de una inicialización particular. Esto es una señal clara de robustez estructural.

2. **Generaliza mejor que las alternativas evaluadas**

   Comparada con las configuraciones 1 y 3, la configuración 2 sostiene de manera más consistente el rendimiento en TEST, con un gap VALID–TEST controlado y sin evidencia de sobreajuste significativo.

3. **Selección final de hiperparámetros**

   Se adopta formalmente la siguiente configuración:

   - `dropout ≈ 0.0847`
   - `lr ≈ 2.46e-4`
   - `weight_decay ≈ 2.2e-5`
   - Arquitectura fija:
     - `d_model = 64`
     - `nhead = 8`
     - `num_layers = 2`
     - `dim_ff = 512`
     - `pooling = "last"`

4. **Cierre metodológico**

   La selección se basa en un proceso completo y formal:

   - Coarse tuning para exploración estructural.
   - Fine tuning para ajuste fino de hiperparámetros.
   - Selección de top-3 configuraciones.
   - Evaluación multi-seed (20 seeds).
   - Análisis de medias, dispersión y gap VALID–TEST.

   En consecuencia, la configuración 2 puede considerarse la versión final robusta del Transformer seq2one para `delta_60` con `L=180`.

## **3.2. Tuneo de MLP**


| cfg_id | R2_valid_mean | R2_valid_std | R2_test_mean | R2_test_std | DA_valid_mean | DA_test_mean | MAE_valid_mean | MAE_test_mean | gap_valid_minus_test_mean |
|-------:|--------------:|-------------:|-------------:|------------:|--------------:|-------------:|---------------:|--------------:|---------------------------:|
| 1 | 0.458434 | 0.012114 | 0.457211 | 0.015802 | 0.746978 | 0.728173 | 21.431548 | 35.983137 | 0.001223 |
| 2 | 0.457837 | 0.010367 | 0.454178 | 0.015404 | 0.745260 | 0.725360 | 21.488390 | 36.238824 | 0.003659 |
| 3 | 0.459992 | 0.013167 | 0.451172 | 0.020257 | 0.746712 | 0.723165 | 21.428539 | 36.469209 | 0.008820 |


La configuración seleccionada es:

**cfg_id = 1**

**Hiperparámetros finales**

- `hidden_dims` = `[512, 256, 128]`
- `activation` = `relu`
- `dropout` ≈ `0.187`
- `lr` ≈ `5.13e-4`
- `weight_decay` ≈ `6.6e-5`
- `window_size` = `180`
- `target` = `delta_60`

**Rendimiento multi-seed**

- R²_valid_mean = 0.458
- R²_test_mean = 0.457
- R²_test_std = 0.016
- gap_valid_test ≈ 0.001

Esto indica:

- **generalización consistente**
- **variabilidad moderada entre seeds**
- **gap VALID–TEST prácticamente nulo**

Por lo tanto, **cfg_id = 1 se adopta como la configuración final del modelo MLP**.

## **3.3. Comparación final: Transformer vs MLP**



Tras completar el proceso de tuning y evaluación multi-seed, se seleccionaron las siguientes configuraciones finales para cada modelo.

**Configuraciones seleccionadas**

| Modelo | Configuración | R2_valid_mean | R2_test_mean | DA_test_mean | MAE_test_mean | Gap (valid-test) |
|------|------|------|------|------|------|------|
| **Transformer** | cfg 2 | 0.586100 | 0.476808 | 0.784349 | 30.365389 | 0.109291 |
| **MLP** | cfg 1 | 0.458434 | 0.457211 | 0.728173 | 35.983137 | 0.001223 |


**1. Capacidad explicativa (R²)**

- **Transformer:** R²_test ≈ **0.477**
- **MLP:** R²_test ≈ **0.457**

El Transformer presenta una **mayor capacidad explicativa del target**, superando al MLP en aproximadamente **2 puntos de R²**.


**2. Directional Accuracy (DA)**

- **Transformer:** DA_test ≈ **0.784**
- **MLP:** DA_test ≈ **0.728**

El Transformer mejora la capacidad de predicción direccional en aproximadamente **5.6 puntos porcentuales**, lo cual es especialmente relevante en aplicaciones de trading.

**3. Error absoluto (MAE)**

- **Transformer:** MAE_test ≈ **30.37**
- **MLP:** MAE_test ≈ **35.98**

El Transformer reduce el error absoluto en aproximadamente **5.6 puntos**, lo que indica una predicción cuantitativa más precisa del movimiento futuro.


**4. Generalización (gap VALID–TEST)**

- **Transformer:** gap ≈ **0.109**
- **MLP:** gap ≈ **0.001**

El MLP muestra un **gap prácticamente nulo**, lo que indica una generalización muy estable entre VALID y TEST.

El Transformer presenta un gap mayor, lo cual sugiere cierta pérdida de rendimiento entre VALID y TEST, aunque **sin deteriorar su superioridad en métricas absolutas**.

**5. Conclusión**

Ambos modelos muestran comportamientos distintos:

**MLP**
- Alta estabilidad entre VALID y TEST.
- Gap prácticamente nulo.
- Desempeño sólido pero menor capacidad predictiva.

**Transformer**
- Mejor rendimiento global en todas las métricas:
  - mayor **R²**
  - mayor **Directional Accuracy**
  - menor **MAE**

A pesar de presentar un gap mayor entre VALID y TEST, el Transformer mantiene **un desempeño significativamente superior al MLP en el conjunto de test**.

**Conclusión final**

El **Transformer (cfg 2)** se posiciona como el **modelo con mejor desempeño global** para la predicción de `delta_60` con `window_size = 180`.

El **MLP (cfg 1)** constituye un baseline robusto y estable, pero con menor capacidad predictiva.

En consecuencia, el **Transformer se adopta como el modelo preferente en esta etapa del pipeline**.